# SOURCE EXPLORATION

## Setup

In [1]:
import os
import requests
import pandas as pd
import duckdb
from edgar import *
from tqdm.auto import tqdm

set_identity("John Doe john.doe@company.com")

In [152]:
PATH_MAIN = os.path.join(os.getcwd(), 'data')
PATH_HOLDINGS = f"{PATH_MAIN}/holdings"
PATH_FINANCIALS = f"{PATH_MAIN}/financials"
os.makedirs(PATH_MAIN, exist_ok=True)
os.makedirs(PATH_HOLDINGS, exist_ok=True)
os.makedirs(PATH_FINANCIALS, exist_ok=True)

In [3]:
ETFS = [
    'XLE', 'XLC', 'XLB', 'XLRE', 'XLU', 'XHB', 'XES', 'XLP', 'XME', 'XTL',
    'XTN', 'XLY', 'XAR', 'XSD', 'XOP', 'KIE', 'XLV', 'XHS', 'XPH', 'KCE',
    'XHE', 'XLK', 'XRT', 'XLF', 'XLI', 'KBE', 'XSW', 'XBI', 'KRE'
]

## Download holdings

In [4]:
def download_holdings(tickers, path=PATH_HOLDINGS):
    headers = {"User-Agent": "Mozilla/5.0"}
    
    for etf in tqdm(tickers):
        
        try:
            ssga_url = f'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-{etf.lower()}.xlsx'
            response = requests.get(ssga_url, headers=headers)
            response.raise_for_status()
            
            # Save and partially clean individual files
            duckdb.sql(f"""
                copy (
                    select
                        '{etf}' as etf
                        , Name as name
                        , Ticker as ticker
                        , Weight::numeric as weight
                    from read_xlsx('{ssga_url}', range = 'A5:H')
                    where true
                        and SEDOL != '-'
                        and ticker != '-'
                ) to '{path}/{etf.lower()}_holdings.parquet'
            """)
            
        except Exception as e:
            print(f"{etf} error: {e}")
            continue

In [5]:
# download_holdings(tickers=ETFS)

In [9]:
holdings = duckdb.sql(f"""
    select * 
    from read_parquet('{PATH_HOLDINGS}/*_holdings.parquet', union_by_name=True)
    order by weight desc
""").fetchdf()

duckdb.sql(f"""
    select etf, count(*) as count, sum(weight) as weight
    from read_parquet('{PATH_HOLDINGS}/*_holdings.parquet', union_by_name=True)
    group by etf
    order by count desc
""").show()

┌─────────┬───────┬───────────────┐
│   etf   │ count │    weight     │
│ varchar │ int64 │ decimal(38,3) │
├─────────┼───────┼───────────────┤
│ KRE     │   160 │        99.898 │
│ XBI     │   150 │        99.821 │
│ XSW     │   132 │        99.958 │
│ KBE     │   103 │        99.915 │
│ XLI     │    81 │        99.956 │
│ XLF     │    76 │        99.820 │
│ XRT     │    75 │        99.812 │
│ XLK     │    74 │        99.927 │
│ XHE     │    67 │        99.891 │
│ KCE     │    65 │        99.941 │
│  ·      │     · │           ·   │
│  ·      │     · │           ·   │
│  ·      │     · │           ·   │
│ XTL     │    40 │        99.786 │
│ XME     │    39 │        99.952 │
│ XHB     │    34 │        99.604 │
│ XES     │    34 │        99.941 │
│ XLP     │    34 │        99.779 │
│ XLRE    │    31 │        99.597 │
│ XLU     │    31 │        99.655 │
│ XLB     │    26 │        99.843 │
│ XLC     │    23 │        99.709 │
│ XLE     │    21 │        99.827 │
└─────────┴───────┴─────────

In [14]:
tickers = holdings.ticker
unique_tickers = list(tickers.unique())
unique_tickers = unique_tickers

print(f'ETFs               {len(holdings.etf.unique())}')
print(f"ETF Holdings       {len(tickers):,.0f}")
print(f"Unique tickers     {len(unique_tickers):,.0f}")

ETFs               29
ETF Holdings       1,767
Unique tickers     1,443


## Download metadata

In [15]:
def download_metadata(etf):
    tickers = holdings[(holdings.etf == etf.upper())].ticker
    metadata = []
    metadata_error = []

    print(f'Downloading {etf.upper()} metadata:')
    for t in tqdm(tickers):
        
        try:
            company = Company(t)
            metadata.append({
                'etf': etf,
                'ticker': t,
                'cik': f"CIK{str(company.cik).zfill(10)}",
                'name': company.name,
                'industry': company.industry,
                'sic': company.sic,
                'fiscal_year': company.fiscal_year_end
            })
    
        except Exception as e:
            metadata_error.append({
                'etf': etf,
                'ticker': t,
                'error': str(e)
            })
    
    metadata = pd.DataFrame(metadata)
    metadata_error = pd.DataFrame(metadata_error)
    
    metadata.to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_holdings_metadata.parquet', index=False)
    
    if not metadata_error.empty:
        metadata_error.to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_holdings_metadata_error.parquet', index=False)

In [16]:
# for e in tqdm(ETFS):
#     download_metadata(etf=e)

## XLE

### Download concepts

In [104]:
def download_concepts(etf):
    tickers = holdings[(holdings.etf == etf.upper())].ticker
    shares_concepts = []
    revenue_concepts = []
    incomeloss_concepts = []
    shares_concepts_error = []
    revenue_concepts_error = []
    incomeloss_concepts_error = []
    check_availability = []

    print(f'Downloading {etf.upper()} concepts:')
    for t in tqdm(tickers):

        try:
            company = Company(t)
            financials = company.get_financials()
            income_stmnt = financials.income_statement()
            income_stmnt = income_stmnt.to_dataframe()

        except Exception as e:
            check_availability.append({
                'tag': 'available', 
                'ticker': t, 'error': str(e)
            })
            continue

        try:
            shares = income_stmnt[income_stmnt.standard_concept == 'SharesAverage'].concept.iloc[0]
            shares_concepts.append({
                'tag': 'shares',
                'ticker': t,
                'concept': str(shares)
            })
            
        except Exception as e:
            shares_concepts_error.append({
                'tag': 'shares',
                'ticker': t,
                'error': str(e)
            })

        try:
            revenue = income_stmnt[income_stmnt.standard_concept == 'Revenue'].concept.iloc[0]
            revenue_concepts.append({
                'tag': 'revenue',
                'ticker': t,
                'concept': str(revenue)
            })
            
        except Exception as e:
            revenue_concepts_error.append({
                'tag': 'revenue',
                'ticker': t, 
                'error': str(e)
            })

        try:
            incomeloss = income_stmnt[income_stmnt.standard_concept == 'NetIncome'].concept.iloc[0]
            incomeloss_concepts.append({
                'tag': 'incomeloss',
                'ticker': t,
                'concept': str(incomeloss)
            })
            
        except Exception as e:
            incomeloss_concepts_error.append({
                'tag': 'incomeloss',
                'ticker': t,
                'error': str(e)
            })

    pd.DataFrame(shares_concepts).to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_concepts_shares.parquet', index=False)
    pd.DataFrame(revenue_concepts).to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_concepts_revenue.parquet', index=False)
    pd.DataFrame(incomeloss_concepts).to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_concepts_incomeloss.parquet', index=False)

    if check_availability:
        pd.DataFrame(check_availability).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_check_availability.parquet', index=False)
    
    if shares_concepts_error:
        pd.DataFrame(shares_concepts_error).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_shares_missing.parquet', index=False)
    
    if revenue_concepts_error:
        pd.DataFrame(revenue_concepts_error).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_revenue_missing.parquet', index=False)
    
    if incomeloss_concepts_error:        
        pd.DataFrame(incomeloss_concepts_error).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_incomeloss_missing.parquet', index=False)

In [105]:
# for e in tqdm(ETFS):
#     download_concepts(etf=e)

# download_concepts(etf='xle')

  0%|          | 0/21 [00:00<?, ?it/s]

### Missing concepts

In [102]:
xle_missing_concepts = duckdb.sql(f"""select * from read_parquet('{PATH_HOLDINGS}/.xle_concepts*', union_by_name=True)""").fetchdf()
xle_missing_concepts

,concept,ticker,error
0,incomeloss,OXY,single positional indexer is out-of-bounds
1,shares,XOM,single positional indexer is out-of-bounds
2,shares,CVX,single positional indexer is out-of-bounds
3,shares,BKR,single positional indexer is out-of-bounds
4,shares,DVN,single positional indexer is out-of-bounds
5,shares,OXY,single positional indexer is out-of-bounds


#### Missing shares concepts fixed by using "us-gaap_EarningsPerShareBasic" / incomeloss

In [61]:
xle_missing_still = []
for t in tqdm(xle_missing_concepts.ticker):
    
    try:
        company = Company(t)
        income_stmnt = company.get_financials().income_statement().to_dataframe()
        eps = income_stmnt[income_stmnt.concept == 'us-gaap_EarningsPerShareBasic'].concept.iloc[0]
    
    except Exception as e:
        xle_missing_still.append({
            'ticker': t,
            'error': str(e)
        })
        continue

if xle_missing_still:
    print(f'Still mising {len(xle_missing_still)}')

  0%|          | 0/6 [00:00<?, ?it/s]

#### Missing OXY incomeloss concept using "us-gaap_ProfitLoss"

In [62]:
oxy = Company('OXY')
oxy = oxy.get_financials().income_statement()
oxy = oxy.to_dataframe()
oxy[oxy.standard_concept.notna()]

,concept,label,standard_concept,2025-12-31 (FY),2024-12-31 (FY),2023-12-31 (FY),level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
1,us-gaap_Revenues,Net sales,Revenue,2.159300e+10,2.201900e+10,2.315600e+10,2,False,False,False,NaN,NaN,NaN,NaN,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
18,us-gaap_SellingGeneralAndAdministrativeExpense,"Selling, general and administrative expense",SellingGeneralAndAdminExpenses,9.860000e+08,9.600000e+08,9.870000e+08,2,False,False,False,NaN,NaN,NaN,NaN,debit,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
19,us-gaap_OtherCostAndExpenseOperating,Other operating and non-operating expense,OtherExpenseIS,1.556000e+09,1.319000e+09,1.165000e+09,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
22,us-gaap_TaxesExcludingIncomeAndExciseTaxes,Taxes other than on income,SellingGeneralAndAdminExpenses,1.030000e+09,1.039000e+09,1.087000e+09,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
24,us-gaap_DepreciationDepletionAndAmortization,"Depreciation, depletion and amortization",DepreciationExpense,7.533000e+09,6.951000e+09,6.449000e+09,2,False,False,False,NaN,NaN,NaN,NaN,debit,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
28,us-gaap_BusinessCombinationAcquisitionRelatedC...,Acquisition-related costs,RestructuringExpenseBenefit,1.300000e+07,8.400000e+07,2.600000e+07,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
29,us-gaap_ExplorationExpense,Exploration expense,ResearchAndDevelopmentExpenses,2.490000e+08,2.750000e+08,4.410000e+08,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
30,us-gaap_InterestAndDebtExpense,"Interest and debt expense, net",InterestExpense,1.079000e+09,1.169000e+09,9.570000e+08,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,oxy_CostsAndOtherDeductions,oxy_CostsAndOtherDeductionsAbstract
39,us-gaap_IncomeLossFromEquityMethodInvestments,Income from equity investments and other,EquityMethodInvestmentIncome,7.600000e+07,7.590000e+08,4.260000e+08,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_NonoperatingIncomeExpense,us-gaap_OtherNonoperatingIncomeExpenseAbstract
42,us-gaap_NonoperatingIncomeExpense,Total,NonoperatingIncomeExpense,7.600000e+07,7.590000e+08,4.260000e+08,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_IncomeLossFromContinuingOperationsBefo...,us-gaap_OtherNonoperatingIncomeExpenseAbstract


# DATA INGESTION

## XLE

### Download fundamentals

In [139]:
xle_catalog = duckdb.sql(f"""
    select *
    from read_parquet('{PATH_HOLDINGS}/xle_holdings_metadata.parquet') 
""").fetchdf()

xle_catalog.head()

,etf,ticker,cik,name,industry,sic,fiscal_year
0,XLE,XOM,CIK0000034088,EXXON MOBIL CORP,Petroleum Refining,2911,1231
1,XLE,CVX,CIK0000093410,CHEVRON CORP,Petroleum Refining,2911,1231
2,XLE,COP,CIK0001163165,CONOCOPHILLIPS,Petroleum Refining,2911,1231
3,XLE,MPC,CIK0001510295,Marathon Petroleum Corp,Petroleum Refining,2911,1231
4,XLE,PSX,CIK0001534701,Phillips 66,Petroleum Refining,2911,1231


### Verified concepts

In [171]:
xle_concepts_verified = duckdb.sql(f"""
    select tag, split_part(concept, '_', 2) as concept, count(*) as count
    from read_parquet('{PATH_HOLDINGS}/xle_concepts*', union_by_name=True)
    group by tag, concept
    order by tag, count desc
""").fetchdf()

# Proxies:
# OXY missing incomeloss --> stock used "us-gaap_ProfitLoss"
# Missing shares         --> "us-gaap_EarningsPerShareBasic" / incomeloss

xle_concepts_verified

,tag,concept,count
0,incomeloss,NetIncomeLoss,20
1,revenue,RevenueFromContractWithCustomerExcludingAssess...,11
2,revenue,Revenues,7
3,revenue,RevenueFromContractWithCustomerIncludingAssess...,3
4,shares,WeightedAverageNumberOfSharesOutstandingBasic,16


In [172]:
xle_incomeloss_concepts = xle_concepts_verified[xle_concepts_verified.tag == 'incomeloss'].concept.tolist()
xle_incomeloss_concepts.append('us-gaap_ProfitLoss')
xle_incomeloss_concepts

['NetIncomeLoss', 'us-gaap_ProfitLoss']

In [173]:
xle_revenue_concepts = xle_concepts_verified[xle_concepts_verified.tag == 'revenue'].concept.tolist()
xle_revenue_concepts

['RevenueFromContractWithCustomerExcludingAssessedTax',
 'Revenues',
 'RevenueFromContractWithCustomerIncludingAssessedTax']

In [174]:
xle_shares_concepts = xle_concepts_verified[xle_concepts_verified.tag == 'shares'].concept.tolist()
xle_shares_concepts

['WeightedAverageNumberOfSharesOutstandingBasic']

In [192]:
def download_financials(tickers, revenue_concepts, incomeloss_concepts, shares_concepts, path=PATH_FINANCIALS):
    os.makedirs(path, exist_ok=True)

    for _, row in tqdm(tickers[["ticker", "cik"]].iterrows()):
        t = row["ticker"]
        c = row["cik"]

        url = f"https://data.sec.gov/api/xbrl/companyfacts/{c}.json"
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        company_facts = response.json()

        for r in revenue_concepts:
            try:
                revenues = company_facts["facts"]["us-gaap"][r]["units"]["USD"]
                df_revenues = pd.DataFrame(revenues)
                duckdb.sql(f"""copy (
                    select '{t}' as ticker, 'revenues' as item, *
                    from df_revenues) to '{path}/{t.lower()}_revenues.parquet'
                """)
                break
            except Exception as e:
                continue

        for i in incomeloss_concepts:
            try:
                income = company_facts["facts"]["us-gaap"][i]["units"]["USD"]
                df_income = pd.DataFrame(income)
                duckdb.sql(f"""copy (
                    select '{t}' as ticker, 'income' as item, *
                    from df_income) to '{path}/{t.lower()}_income.parquet'
                """)
                break
            except Exception as e:
                continue

        for s in shares_concepts:
            try:
                shares = company_facts["facts"]["us-gaap"][s]["units"]["shares"]
                df_shares = pd.DataFrame(shares)
                duckdb.sql(f"""copy (
                    select '{t}' as ticker, 'shares' as item, * 
                    from df_shares) to '{path}/{t.lower()}_shares.parquet'
                """)
                break
            except Exception as e:
                continue

In [193]:
download_financials(
    tickers=xle_catalog,
    revenue_concepts=xle_revenue_concepts,
    incomeloss_concepts=xle_incomeloss_concepts,
    shares_concepts=xle_shares_concepts,
    path=f'{PATH_FINANCIALS}/xle'
)

0it [00:00, ?it/s]

# DATA PROCESSING

## XLE

In [196]:
xle_revenues = duckdb.sql(f"""
    select *
    from read_parquet('{PATH_FINANCIALS}/xle/*revenues.parquet', union_by_name=True)
""")

┌─────────┬──────────┬────────────┬────────────┬──────────────┬──────────────────────┬────────┬─────────┬─────────┬────────────┬──────────┐
│ ticker  │   item   │   start    │    end     │     val      │         accn         │   fy   │   fp    │  form   │   filed    │  frame   │
│ varchar │ varchar  │  varchar   │  varchar   │    int64     │       varchar        │ double │ varchar │ varchar │  varchar   │ varchar  │
├─────────┼──────────┼────────────┼────────────┼──────────────┼──────────────────────┼────────┼─────────┼─────────┼────────────┼──────────┤
│ APA     │ revenues │ 2020-01-01 │ 2020-06-30 │            0 │ 0001871133-21-000007 │ 2021.0 │ Q2      │ 10-Q    │ 2021-08-05 │ NULL     │
│ APA     │ revenues │ 2021-01-01 │ 2021-03-31 │            0 │ 0001673379-21-000012 │ 2021.0 │ Q1      │ 10-Q    │ 2021-05-07 │ CY2021Q1 │
│ APA     │ revenues │ 2021-01-01 │ 2021-06-30 │            0 │ 0001871133-21-000007 │ 2021.0 │ Q2      │ 10-Q    │ 2021-08-05 │ NULL     │
│ APA     │ revenues

In [ ]:
# df_xle_shares = duckdb.sql(f"""
#     select 
#         '{t}' as ticker
#         , 'Shares' as item
#         , form
#         , filed::date as date
#         , "end"::date as period_end
#         , val::bigint as shares
#     from read_parquet
#     where (filed, "end") in (
#         select filed, max("end")
#         from df_shares
#         group by filed
#     )
#     order by filed desc
# """).show()